[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/validation/labse-multilingual-smoketest/experiments/labse_he_en_colab_validation.ipynb)

Open this exact notebook in Colab, connect to a runtime, and run cells from the top using ▶. GitHub Preview displays the notebook without executing it.


# LaBSE — Hebrew–English employer-name equivalence probe

**Purpose:** Test whether the *pretrained* LaBSE model assigns higher cross-lingual similarity to corresponding organization names than to unrelated organizations. No retraining, paid API, token, GPU, or manually maintained dictionary is required. Run every cell in **Google Colab (free CPU)**.

**Limitations:** LaBSE is trained for cross-language sentence representations, **not** verified employer entity resolution. This notebook reports observed cosine similarities and nearest neighbors. A high score is not proof that names refer to the same legal entity. Real data integration requires validation. The first execution downloads publicly available model weights (~1.9 GB) and requires Internet access. No model inference result is claimed until this notebook executes.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2"])

import torch
from sentence_transformers import SentenceTransformer
import numpy as np

print("PyTorch:", torch.__version__)
print("Device: CPU (no GPU required)")

In [ ]:
from IPython.display import HTML, display
from html import escape
import numpy as np

# Data examples are the original test inputs. Extra Hebrew forms below are
# diagnostic candidates, NOT additional observed values from the survey.
original_labels = [
    "Intel", "אינטל", "NVIDIA", "נבידיה",
    "Microsoft", "מיקרוסופט", "bird", "birds", "ציפור",
    "קרית גת", "קריית גת", "בית ספר יסודי", "primary school"
]
alternate_nvidia_spellings = ["אנבידיה", "אנווידיה"]
labels = original_labels + alternate_nvidia_spellings

torch.set_num_threads(2)
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
vectors = model.encode(
    labels, batch_size=4, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False
)
assert vectors.shape == (len(labels), 768), vectors.shape
assert np.isfinite(vectors).all()
positions = {label: i for i, label in enumerate(labels)}

def cosine(a, b):
    return float(vectors[positions[a]] @ vectors[positions[b]])

def show_table(heading, rows, caption=None):
    html_rows = []
    for left, right, description in rows:
        score = cosine(left, right)
        html_rows.append(
            "<tr>"
            f"<td>{escape(left)}</td><td>{escape(right)}</td>"
            f"<td style='text-align:right;font-variant-numeric:tabular-nums'>{score:.4f}</td>"
            f"<td>{escape(description)}</td>"
            "</tr>"
        )
    table = (
        f"<h3>{escape(heading)}</h3>"
        "<table style='border-collapse:collapse;width:100%;font-size:14px'>"
        "<thead><tr>"
        "<th style='text-align:left'>Name A</th><th style='text-align:left'>Name B</th>"
        "<th style='text-align:right'>Cosine similarity</th>"
        "<th style='text-align:left'>Known relationship</th>"
        "</tr></thead><tbody>"
        + "".join(html_rows) +
        "</tbody></table>"
    )
    if caption:
        table += f"<p><small>{escape(caption)}</small></p>"
    display(HTML(table))

display(HTML(
    "<h2>LaBSE: cross-language name matching</h2>"
    f"<p>Successfully encoded <b>{len(labels)} names</b> into "
    f"<b>{vectors.shape[1]}-dimensional vectors</b> using pretrained PyTorch LaBSE."
    " Similarity is a cosine score, <b>not a probability</b> of identity.</p>"
))

show_table("1. Known equivalent names", [
    ("Intel", "אינטל", "Same company"),
    ("Microsoft", "מיקרוסופט", "Same company"),
    ("NVIDIA", "נבידיה", "Same company; spelling present in survey")
], "These are positive examples. The model's scores do not define a universal match threshold.")

show_table("2. Different names and linguistic controls", [
    ("NVIDIA", "Intel", "Different companies, similar industry"),
    ("Intel", "בית ספר יסודי", "Unrelated entities"),
    ("bird", "birds", "Same concept, singular/plural"),
    ("bird", "ציפור", "Translation"),
    ("קרית גת", "קריית גת", "Same city, spelling variants"),
    ("בית ספר יסודי", "primary school", "Translation")
])

show_table("3. Why is NVIDIA lower? Test alternative Hebrew spellings", [
    ("NVIDIA", "נבידיה", "Survey spelling"),
    ("NVIDIA", "אנבידיה", "Alternative Hebrew spelling; not in survey test data"),
    ("NVIDIA", "אנווידיה", "Alternative Hebrew spelling; not in survey test data")
], "Compare these measured scores to test a spelling-sensitivity hypothesis. A difference alone does not establish the model's internal cause.")

nvidia_alias = cosine("NVIDIA", "נבידיה")
hard_negative = cosine("NVIDIA", "Intel")
candidates = [(spelling, cosine("NVIDIA", spelling)) for spelling in ["נבידיה", "אנבידיה", "אנווידיה"]]
top_spelling, top_score = max(candidates, key=lambda x: x[1])

display(HTML(
    "<h3>4. What the test actually establishes</h3>"
    f"<p>The provided Hebrew alias for NVIDIA scored <b>{nvidia_alias:.4f}</b>, "
    f"compared with <b>{hard_negative:.4f}</b> against Intel, a different company. "
    f"The observed margin is <b>{nvidia_alias-hard_negative:+.4f}</b>.</p>"
    f"<p>The closest of the three tested Hebrew spellings is "
    f"<b>{escape(top_spelling)}</b> (score <b>{top_score:.4f}</b>). "
    "A higher similarity for another spelling would support spelling sensitivity; "
    "it would <b>not</b> prove a particular cause or guarantee correct entity resolution.</p>"
    "<p><b>Decision:</b> This is a diagnostic experiment, not a verified production "
    "matching algorithm. Do not automatically merge employer names from these scores alone.</p>"
))
print("MODEL_EXECUTION_SUCCESS: All similarities above were measured during this run.")


## Broader test: is cosine ≥ 0.50 safe for employer-name identity?

The pairs below are **hand-labelled diagnostic examples**, not a representative sample of the survey. Positive pairs mean the same named entity; negative pairs are deliberately challenging (same industry, similar spellings, overlapping words, or semantically related but different entities). No training or paid service is involved. The model already loaded above is reused.

The test prints all false positives and false negatives at 0.50, a confusion matrix, precision and recall, and a threshold sweep. **This is a screening experiment, not statistical validation of the 0.50 rule on your survey.** A production decision requires verified labels from the actual employer-name distribution, especially where similar names refer to different entities.

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Diagnostic positive aliases: the same named entity, including transliteration.
positive_pairs = [
    ("Intel", "אינטל"),
    ("Microsoft", "מיקרוסופט"),
    ("NVIDIA", "נבידיה"),
    ("NVIDIA", "אנווידיה"),
    ("Google", "גוגל"),
    ("Apple", "אפל"),
    ("Amazon", "אמזון"),
    ("Facebook", "פייסבוק"),
    ("Samsung", "סמסונג"),
    ("Netflix", "נטפליקס"),
    ("IBM", "איי בי אם"),
    ("Tesla", "טסלה"),
    ("Oracle", "אורקל"),
    ("Adobe", "אדובי"),
    ("Spotify", "ספוטיפיי"),
    ("Siemens", "סימנס"),
    ("Walmart", "וולמארט"),
    ("Disney", "דיסני"),
    ("PayPal", "פייפאל"),
    ("LinkedIn", "לינקדאין"),
]

# Diagnostic negative pairs: explicitly distinct companies, places or generic labels.
# Some are related semantically or lexically; that must not imply identical entities.
negative_pairs = [
    ("Intel", "NVIDIA"), ("Intel", "AMD"), ("Intel", "Intuit"),
    ("Intel", "Intellia Therapeutics"), ("Microsoft", "MicroStrategy"),
    ("Microsoft", "Google"), ("Microsoft", "Apple"),
    ("NVIDIA", "NIVEA"), ("NVIDIA", "AMD"), ("NVIDIA", "Intel"),
    ("Google", "GoPro"), ("Google", "Apple"), ("Apple", "Apple Bank"),
    ("Amazon", "Amazon River"), ("Amazon", "Alibaba"),
    ("Meta", "MetLife"), ("Meta", "Microsoft"),
    ("Tesla", "Tesla Bank"), ("Tesla", "Toyota"),
    ("Oracle", "Oracle Health"),  # product/business unit vs parent label: NOT a validated different legal entity
    ("Adobe", "Adidas"), ("Adobe", "Autodesk"),
    ("Netflix", "NetApp"), ("Netflix", "Disney"),
    ("Samsung", "Sony"), ("Spotify", "SoundCloud"),
    ("PayPal", "Payoneer"), ("PayPal", "PayPal Credit"), # related brand: ambiguous as legal entity
    ("בנק לאומי", "בנק הפועלים"),
    ("אוניברסיטת תל אביב", "אוניברסיטת חיפה"),
    ("בית ספר יסודי", "בית ספר תיכון"),
    ("מסעדה", "בית קפה"),
    ("חברת החשמל", "חברת המים"),
    ("תל אביב", "תל אביב יפו"), # overlapping place reference: ambiguous; exclude from scored negatives
    ("Intel", "בית ספר יסודי"),
]

# Exclude ambiguous corporate parent/product or overlapping-place relationships
# rather than presenting them as established distinct entities.
ambiguous_pairs = {
    ("Oracle", "Oracle Health"), ("PayPal", "PayPal Credit"),
    ("תל אביב", "תל אביב יפו")
}
negative_pairs = [p for p in negative_pairs if p not in ambiguous_pairs]

examples = [(a, b, 1) for a, b in positive_pairs] + [(a, b, 0) for a, b in negative_pairs]
unique_names = list(dict.fromkeys(x for a, b, _ in examples for x in (a, b)))
emb = model.encode(unique_names, batch_size=8, normalize_embeddings=True,
                   convert_to_numpy=True, show_progress_bar=False)
lookup = {x: emb[i] for i, x in enumerate(unique_names)}
rows = [{"left": a, "right": b, "same_entity": y,
         "cosine": float(lookup[a] @ lookup[b])} for a, b, y in examples]
results = pd.DataFrame(rows)
threshold = 0.50
results["predicted_same"] = (results["cosine"] >= threshold).astype(int)
results["error_type"] = np.select(
    [(results.same_entity == 0) & (results.predicted_same == 1),
     (results.same_entity == 1) & (results.predicted_same == 0)],
    ["FALSE POSITIVE (unsafe merge)", "FALSE NEGATIVE (missed alias)"],
    default="correct"
)
tn, fp, fn, tp = confusion_matrix(results.same_entity, results.predicted_same, labels=[0, 1]).ravel()
print(f"TEST: {len(results)} labelled diagnostic pairs, {len(positive_pairs)} positive, {len(negative_pairs)} negative")
print(f"RULE: cosine >= {threshold:.2f} -> SAME ENTITY")
print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"Precision={precision_score(results.same_entity,results.predicted_same,zero_division=0):.3f}, "
      f"Recall={recall_score(results.same_entity,results.predicted_same,zero_division=0):.3f}")
print("\nFALSE POSITIVES: distinct names incorrectly accepted as the same:")
display(results[results.error_type.str.startswith("FALSE POSITIVE")].sort_values("cosine",ascending=False))
print("\nFALSE NEGATIVES: same-entity aliases missed:")
display(results[results.error_type.str.startswith("FALSE NEGATIVE")].sort_values("cosine"))
print("\nALL HARD NEGATIVES ranked by score:")
display(results[results.same_entity.eq(0)].sort_values("cosine",ascending=False))
print("\nTHRESHOLD SWEEP (diagnostic examples only):")
sweep = []
for t in [0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90]:
    p=(results.cosine>=t).astype(int)
    tn_t,fp_t,fn_t,tp_t=confusion_matrix(results.same_entity,p,labels=[0,1]).ravel()
    sweep.append({"threshold":t,"TP":tp_t,"FP":fp_t,"FN":fn_t,"TN":tn_t,
                  "precision":round(precision_score(results.same_entity,p,zero_division=0),3),
                  "recall":round(recall_score(results.same_entity,p,zero_division=0),3)})
display(pd.DataFrame(sweep))
print("\nImportant: even zero false positives in this small constructed sample would not validate a universal 0.50 threshold.")


## Additional test using the 82 exact employer-name values in the assignment workbook

This separate, self-contained test uses **all 82 distinct `ShemAvoda` strings** observed in the 200-row survey workbook, including their original spacing and spelling, and evaluates all **3,321** possible unordered pairs. It additionally checks 6 known employer aliases, 35 *distinct* organization pairs, and 10 ambiguous generic/specific pairs whose identity cannot be inferred safely from their names. The arrays below are a snapshot of the observed survey values, not a claim that new survey records are available.

The test measures the proposed **cosine > 0.50** rule alongside higher thresholds. It reports false merges, missed aliases, and the highest-similarity pairs among all 82 names. All-pairs ranking is **unlabelled candidate discovery**, not a valid precision estimate. The 6/35 hand-reviewed pairs are deliberately selected examples and cannot prove a population-wide error rate. The KPMG/סומך חייקין relationship is documented by KPMG Israel: https://www.kpmg.co.il/תנאי-האתר . No API key, GPU, training, manual dictionary, or paid provider is required.

In [ ]:
from itertools import combinations
from IPython.display import display, HTML
from html import escape
import numpy as np

employers = ["מסעדה","משרד החינוך","בית ספר תיכון","בנק לאומי","משטרה","אינטל","סופר פארם","קניון עזריאלי","למ\"ס","סולל בונה","קוסמטיקאית עצמאית","NVIDIA","אוסם","אמדוקס","בית חולים הדסה עין כרם","בית מלון","בנק מזרחי","גידול תפוזים ","משרד ממשלתי","נבידיה","שופר סל","EMILIO","חברת הייטק","מקורות","עירייה","תיאטרון ירושלים","COGNYTE","INTEL","א.א. סיעוד","אוניברסיטת תל אביב","אחים כהן","אמיר כהן","מלון דן","עובד במפעל  ","עובד במפעל של הקיבוץ","עיריית חדרה","עיריית ירושלים","עיריית תל אביב","עיריית תל אילת","עלית","רמי לוי","שופרסל","4 עונות","KPMG","MICROSOFT","אדרים","אוניברסיטת באר שבע","איחים חג'ג'","בזק","בנק אוצר החייל","בנק דיסקונט","בנק הפועלים","גידול תפוחים ","גן חדרה ","דואר ישראל","הרודס","חח\"י","חנות בגדים של מעצבים","חנות פולגת","יועצץ מחשוב","יועצת IT עצמאית","ישרוטל","מוסך א.א.","מוסך אחים.","מיקרוסופט","מכללת חדרה","מלון אקאדיה","מסעדת טעמים","מסעדת נונה","מסעדת עזורה","משרד הפנים","סומך חייקין","סופרפארם","עובד במפעל   הבקבוקים","פרשקובסקי","קוסמטיקאית ","קריית הממשלה","רשת בי פארם","תיאטרון ","תיאטרון הבימה","תיאטרון חיפה","תל אביב יפו - מסעדה"]
known_same = [["INTEL","אינטל"],["NVIDIA","נבידיה"],["MICROSOFT","מיקרוסופט"],["שופר סל","שופרסל"],["סופר פארם","סופרפארם"],["KPMG","סומך חייקין"]]
known_different = [["בנק לאומי","בנק מזרחי"],["בנק לאומי","בנק דיסקונט"],["בנק לאומי","בנק הפועלים"],["בנק מזרחי","בנק דיסקונט"],["בנק הפועלים","בנק דיסקונט"],["בנק הפועלים","בנק מזרחי"],["משרד החינוך","משרד הפנים"],["עיריית תל אביב","עיריית ירושלים"],["עיריית תל אביב","עיריית חדרה"],["עיריית ירושלים","עיריית חדרה"],["אוניברסיטת תל אביב","אוניברסיטת באר שבע"],["אוניברסיטת תל אביב","עיריית תל אביב"],["תיאטרון חיפה","תיאטרון ירושלים"],["תיאטרון חיפה","תיאטרון הבימה"],["תיאטרון ירושלים","תיאטרון הבימה"],["מסעדת נונה","מסעדת עזורה"],["מסעדת נונה","מסעדת טעמים"],["מסעדת טעמים","מסעדת עזורה"],["מלון דן","מלון אקאדיה"],["סופר פארם","רשת בי פארם"],["סופרפארם","רשת בי פארם"],["שופר סל","רמי לוי"],["שופרסל","רמי לוי"],["אוסם","עלית"],["אינטל","נבידיה"],["INTEL","NVIDIA"],["NVIDIA","MICROSOFT"],["אמדוקס","מיקרוסופט"],["MICROSOFT","COGNYTE"],["סופר פארם","שופרסל"],["מוסך א.א.","א.א. סיעוד"],["בית ספר תיכון","בית חולים הדסה עין כרם"],["אוניברסיטת תל אביב","מכללת חדרה"],["מלון דן","ישרוטל"],["קניון עזריאלי","אוניברסיטת תל אביב"]]
unknown_identity = [["מסעדה","מסעדת נונה"],["מסעדה","מסעדת טעמים"],["בית מלון","מלון דן"],["משרד ממשלתי","משרד הפנים"],["עירייה","עיריית ירושלים"],["חברת הייטק","NVIDIA"],["עובד במפעל  ","אינטל"],["תיאטרון ","תיאטרון חיפה"],["קוסמטיקאית עצמאית","קוסמטיקאית "],["עיריית תל אילת","עיריית תל אביב"]]

assert len(employers) == len(set(employers)) == 82
assert len(known_same) == 6 and len(known_different) == 35 and len(unknown_identity) == 10

def ordered(a, b):
    return tuple(sorted([a, b]))

labels_same = {ordered(*p) for p in known_same}
labels_different = {ordered(*p) for p in known_different}
labels_unknown = {ordered(*p) for p in unknown_identity}
assert (not labels_same & labels_different and not labels_same & labels_unknown
        and not labels_different & labels_unknown)

# Uses already downloaded/launched model from earlier cells: no second model.
all_tokens = sorted(set(employers))
feature_vectors = model.encode(all_tokens, batch_size=8, normalize_embeddings=True,
                               convert_to_numpy=True, show_progress_bar=False)
assert feature_vectors.shape == (82, 768) and np.isfinite(feature_vectors).all()
ids = {name: i for i, name in enumerate(all_tokens)}
def score(a, b):
    return float(feature_vectors[ids[a]] @ feature_vectors[ids[b]])

def render(title, headers, items, subtitle=""):
    h = "".join(f"<th style='text-align:left;padding:6px;border-bottom:1px solid #8886'>{escape(str(x))}</th>" for x in headers)
    body = "".join("<tr>"+"".join(
        f"<td style='padding:5px;border-bottom:1px solid #8882'>{escape(str(x))}</td>"
        for x in row)+"</tr>" for row in items)
    display(HTML(f"<h3>{escape(title)}</h3><table style='width:100%;font-size:13px;border-collapse:collapse'>"
                 f"<thead><tr>{h}</tr></thead><tbody>{body}</tbody></table>"
                 + (f"<p><small>{escape(subtitle)}</small></p>" if subtitle else "")))

data = [(a,b,score(a,b),1) for a,b in known_same] + [
    (a,b,score(a,b),0) for a,b in known_different]
summary = []
for t in (0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90):
    tp=sum(y==1 and s>t for a,b,s,y in data)
    fp=sum(y==0 and s>t for a,b,s,y in data)
    fn=sum(y==1 and s<=t for a,b,s,y in data)
    tn=sum(y==0 and s<=t for a,b,s,y in data)
    precision = f"{tp/(tp+fp):.1%}" if tp+fp else "undefined"
    recall = f"{tp/(tp+fn):.1%}" if tp+fn else "undefined"
    summary.append([f"> {t:.2f}", tp, fp, fn, tn, precision, recall])

display(HTML(
    "<h2>Employer names from the actual survey: can >0.50 be used as an automatic merging rule?</h2>"
    "<p>82 unique values, 3,321 possible pairs; 6 reviewed positives, "
    "35 reviewed negatives, 10 ambiguous pairs excluded from metrics. "
    "Scores are cosine similarity, not identity probabilities.</p>"
))
render("1. Threshold sweep on reviewed employer pairs",
       ["Rule","TP (same)","FP (unsafe merges)","FN (missed)","TN","Precision*","Recall*"],
       summary,
       "*Only on this hand-selected benchmark, not a real-world error-rate estimate.")

false_merges = sorted([(a,b,f"{s:.4f}") for a,b,s,y in data if y==0 and s>0.50],
                      key=lambda r:float(r[2]),reverse=True)
missed_aliases = sorted([(a,b,f"{s:.4f}") for a,b,s,y in data if y==1 and s<=0.50],
                        key=lambda r:float(r[2]))
hard_negatives = sorted([(a,b,f"{s:.4f}") for a,b,s,y in data if y==0],
                        key=lambda r:float(r[2]),reverse=True)
render("2. False positives: different employers incorrectly merged above 0.50",
       ["Employer A","Employer B","Similarity"],
       false_merges or [("No false positive within these 35 pairs","","")],
       f"{len(false_merges)} false positives in the 35 selected distinct-employer pairs.")
render("3. False negatives: known aliases missed at 0.50",
       ["Alias A","Alias B","Similarity"],
       missed_aliases or [("None within six tested positive pairs","","")])
render("4. Highest-scoring truly different employers (top 25)",
       ["Employer A","Employer B","Similarity"],hard_negatives[:25])

unknown_scored=sorted([(a,b,f"{score(a,b):.4f}") for a,b in unknown_identity],
                      key=lambda r:float(r[2]),reverse=True)
render("5. Generic/specific labels: identity unknown — do not merge automatically",
       ["Label A","Label B","Similarity"],unknown_scored)

all_pairs=sorted([(a,b,score(a,b)) for a,b in combinations(all_tokens,2)],
                 key=lambda r:r[2],reverse=True)
over_half=sum(s>0.50 for _,_,s in all_pairs)
render("6. All 82 employer strings: top 30 candidate pairs for review",
       ["Employer A","Employer B","Similarity"],
       [(a,b,f"{s:.4f}") for a,b,s in all_pairs[:30]],
       f"{over_half}/{len(all_pairs)} possible pairs score strictly > 0.50. "
       "Most pairs are unlabelled; do not call them true or false matches.")

display(HTML(
    "<h3>7. Decision</h3>" +
    ("<p><b>Do not use >0.50 for automatic merging.</b> "
     f"{len(false_merges)} of 35 distinct-employer examples would be merged incorrectly.</p>"
     if false_merges else
     "<p><b>Universal automatic merging is not validated.</b> "
     "The hand-picked negatives happen not to exceed 0.50, but the sample "
     "does not estimate false-positive rates on new data.</p>") +
    "<p>Use LaBSE to retrieve candidate name pairs; require additional "
    "identity verification before changing employer labels.</p>"
))
print("SURVEY_THRESHOLD_EXPERIMENT_SUCCESS")


## Domain-adaptive LaBSE | occupations-relevant workplace sectors — not company identity

**Aim.** Compare original LaBSE with **fine-tuning of its existing Dense projection layer**, using industry-code evidence from the simulated survey. The **BERT Transformer is frozen** to make this experiment feasible on a free Google Colab CPU; this is genuine parameter fine-tuning of LaBSE's final Dense module, **not** full-BERT fine-tuning. The objective is **workplace-sector similarity**, a potential feature for occupation classification, **not** asserting that banks or restaurants with different names are the same company or that employees of one bank share the same occupation.

**Ground truth / limitations.** Supervision is derived exclusively from the first two digits of `SemelAnafSofi` (economic industry), not `SemelMishlachSofi` (the occupational classification target). For a given employer name, conflicting or unknown industry codes cause exclusion **from fine-tuning**, while retaining the name for clustering. Validation names are never used for training. Industry-code groupings can be noisy because the source is simulated and manually coded, and **sector similarity is not occupational identity**. We report improvement or deterioration on held-out industry labels without selecting a threshold on validation. The user's fixed decision threshold is **cosine > 0.50**.

**Self-contained data.** This notebook embeds a compressed, SHA-256-checked snapshot of the original 200-record workbook (ID, ShemAvoda, SemelAnafSofi, SemelMishlachSofi). No Drive authentication, file upload, Hugging Face token, paid API, or external dataset is required. The original survey stays unchanged. The already-downloaded pretrained model is reused.

In [ ]:

# Free Colab usually includes these packages. Install ONLY missing dependencies to
# avoid replacing live NumPy/PyTorch binaries and triggering runtime restart.
import importlib.util, subprocess, sys
requirements = {"plotly":"plotly>=5.24,<7", "matplotlib":"matplotlib>=3.8,<4",
                "sklearn":"scikit-learn>=1.4,<2", "pandas":"pandas>=2,<3",
                "PIL":"pillow>=10"}
missing = [requirement for module,requirement in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])

import base64, zlib, json, hashlib, re, copy
from collections import defaultdict, Counter
from itertools import combinations
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import OneHotEncoder
from IPython.display import HTML, display

DATA_B64 = "eNqlmelu20YQgF+F8J/+cYudvfdnkaaB0SQuWqNwkeadrEiKrESWDzjx8UJ8nc6xlESJpFZO4sSWd/aY69sZ8sMHOD6qF/Xyv6P69uj4KFoA/KZBq6OPxx80Do7qef1c1bN6inLz+qqqJ/W4vq0nVX2Hn6+r+mv9WF/RZA8KvxmDa+Bkg5MfcGguC0zqJf59wkUm642MA02yFmVp8LEek+QMp9zghl9IUvGqzoIlSYeSFyg7rx9p50l9jYPn+IdkjObVfF7tEteTzbSRJXg48BK0w2W9IG3zuUFmx63hoBXbJB814fATnu2eDnDLEjawMk4bthqo46PX707enpziL/1qNh8fgE16Uz9UaIkLXAdVETEt9jAsptdiy/oz6jETMQuJT2zYxGBY01v0xJg1daIKriLjZNcbduEY/01kKklELdoA2vP9Pye/nfwqhzAbmgLZ8bZR9R5PK362IWhWV8tZAzt6iqJLNNkcf74Ql9/V3/FX/AHlk2cV0RCyetz0eRMbz6xmjg4gt5Io2ZyCbRVM9OOUJ8wl9JxY2cjaWm1PeK4XFZ6ELDEi+ZiUGMOxPLDPSQVaDVSwpKcRZ2idR29oNq55i98vu1Z1jlfN1teGTzHFcdRwhBre40ec8UT/TdBOtNy0/iZbkgIRomxJjpvxZpw6c97xoRVPsgflAxoNT3HJNrneNU5SSvJS/KX9Oqtvxbe0wFeazEfQtEECCQId1tJLXFKEnOM4WwnFvjjMHEkcImQ/PuEdhzQdesH+xohSGxFlyH2fOGinZKpnPOUUpxB9UEwBp3PSmTOQw3RBsiP204Tdx2Ihg8PoIdIZ8yLUNUcQflESkOkp2o1nowfQniVcMeGM53CbyQl6EGfCMOMMeuTk/dnrt7t8M2kYcFatCdcNOAu8+ZzxPs2IuFxTrIGd1T2wa1PMmiLYWdsdZCmrbF33+HmmiPV9MExGJ5YI/TC0sQiGNr0Ihk4Vw9DBgTB0+jAYOsHWIp8UyWQ2YOjsy2DoXDcMq134OXTUq9M37/89e70LPBeKgae9E4XiIcBzqQB4Xg0Dz8NBwPP6EOB50wu8IMgJLGaHgOfdjwDPyx3yWa4kkLGcKT4Uo87Hkmou7SnnVDfPGt4FGOZd0HsrumD6K7pgiyq64IogF/xwRRfCvoouxA2I5bGVqqkIYlG9CGIRiiEW+5g0pxm7DItmn3ybYdEOFXTR9TFsRCahAm1U3+0yLPoXF3QxFBZ0MR5c0MV0CN+SKuBbgmG+JX0Q35I5hG/JFhV0yXXxzWqhX/I/wrcU+go6nYM9xWLKpVRQ0IFSw5gDBb0lHSjd3dOuWjllCtpWO9C3KldUy4HyZZ2rCsPVHKg4XM6BSjsozGRv6jmgTnzFwi3sA0BZewv6Zf1tbs6LGlxu1Ft8m6FSOQ7aKARwA7JtDAL4oVoOQJzwlTJte1PLlTw0krG4fANq13vrN+DefAtwlBr3vO9W6QYaDmEbaF0AN9BmmG7A7Xc53oC78GK+gfZFBRzo8koKuAW/+AW/Kr7OyFfjDszo1F7znrW82ngkZ5ysaNT2Hcr9kG0ePcEeXnHDXXh6br+p33ziTBvhng9i6Tbzwdg98ZAd6cQx3HbnHN9Nca19PqpvpRRrg2xla5gsEnKpSi5+4BvHbwHFxD4R1E8CK/feXSLNAzluvynuiC7P1cnZNmfWTXZ+xkeOIMtx/TKvBvS0uq+1X1GVmvF3J6/+Ov379PezdRuMmZV3s30aNPU/UCP+x5/v3tBQ0q3k5x58K/kn7M9lZ/rbcFD6W7kzFrLwuDP5uUPf7FUrhs+0+rkaZIJThzGB+/QdJsw2mGA3mcBdehcTgO0fG/o7M5BVDs452qxYT3r1/Uxwuw8RO3zhDnpuCC5s+uKCb83mnt72idu865/FttOMlR0/HPYsEbxgbMxCVx2U9VDOKa93QbF15XJ3/kjVUcXKzzsqLd9UuMv6i9x+c24erjdSxubA8K6LTZnUkrPcjTfI+EZhPOMsH3XQwodtqsMmX33cV/iCT+sKjxb6TpEwyVosWp2vTOAGfU/vkyU33o5MOF/uVk1PU0Da5q0DusJWHNM3+aSYK6yMy3EdzGDNapvcC0Ovn5w+5+oruy64wpci/mVVIzf4fYV6o1fcH2Ih9aJwlUqxeXKSW5NZ/ekn/FpbEm8/CQtu7Wdi6Iods0C2PXekW9Rl9ok9L0cquRAeGIEP2fdGgLWqMOPGMxdiyi1b+kaeQTdxkmMqurIg4OZ+peII1RuLVbi5X7LbRg1CtpWW7n6GrWi7aMnDklQUC5juDGSSco1UPmgaPKgxWY4b+Amn55jv38atJMcC+yqzpNu4vcQfllm17TeI3LrTFcYPRuacJBcclc60+7FkW1pKUHUpyo27rDTN7eoqMptd/T4VQluFm9WFuaNA7EnEjuTjTv2TdCTcCVVd7xZV53MHdvbH/wEo+MX8"
DATA_SHA256 = "82a7787f16f0dfecbbf0002156b70f30459a369097b5835bf9c65086fbea2d44"
raw = zlib.decompress(base64.b64decode(DATA_B64))
assert hashlib.sha256(raw).hexdigest() == DATA_SHA256, "Survey snapshot integrity check failed"
survey = pd.DataFrame(json.loads(raw), columns=["ID","ShemAvoda","SemelAnafSofi","SemelMishlachSofi"])
assert len(survey)==200 and survey.ID.is_unique and survey.ShemAvoda.nunique()==82
assert sorted(survey.ID.tolist()) == list(range(1,201))

THEME = {
    "paper":"#F8F7F1","ink":"#253D31","green":"#234737",
    "muted":"#59695F","rule":"#DCE0D4","sage":"#E9EDE2",
    "series":["#234737","#946E37","#507589","#9F5550","#735C80","#3E8781"]
}
DARK = {"paper":"#14261F","ink":"#F8F7F1","rule":"#3C5647","sage":"#1B3329"}
BASE_SOURCE = "theme/portfolio/palette.json · shulruferor-sudo/Advance-Data-Science"
OUT = Path("/content/labse_occupation_outputs")
OUT.mkdir(parents=True, exist_ok=True)
np.random.seed(2026)
torch.manual_seed(2026)
torch.set_num_threads(2)
print(f"Verified snapshot: {len(survey)} records, {survey.ShemAvoda.nunique()} distinct names.")
print("Colors sourced from",BASE_SOURCE)
display(survey.head(5))


In [ ]:
# Reproducible name-disjoint sector-proxy fine-tuning (LaBSE Dense only).
names = sorted(survey.ShemAvoda.unique().tolist())
name_codes = defaultdict(set)
for row in survey.itertuples(index=False):
    code = str(row.SemelAnafSofi)
    name_codes[row.ShemAvoda].add(code[:2] if len(code)==4 and code.isdigit() else "UNKNOWN")
stable_sector = {name: next(iter(codes)) for name,codes in name_codes.items()
                 if len(codes)==1 and "UNKNOWN" not in codes}
uncertain_names = sorted(set(names)-set(stable_sector))
sector_members = defaultdict(list)
for name,sector in stable_sector.items():
    sector_members[sector].append(name)

rng = np.random.default_rng(2026)
holdout = []
for sector,group in sorted(sector_members.items()):
    if len(group) >= 4:
        holdout.extend(rng.choice(sorted(group), size=2, replace=False).tolist())
holdout = sorted(holdout)
train_names = sorted(set(stable_sector)-set(holdout))
assert len(set(train_names)&set(holdout))==0 and len(holdout)>=8
print(f"Stable economic-sector names: {len(stable_sector)} / 82.")
print(f"Excluded from weak-label training (conflicting/unknown codes): {len(uncertain_names)}")
print(f"Train-only employer names: {len(train_names)} | held-out names: {len(holdout)}")

# First run model.encode on all names (NO weights changed yet).
model.eval()
baseline_vectors = model.encode(names, batch_size=8, device="cpu",
                                normalize_embeddings=True, convert_to_numpy=True,
                                show_progress_bar=False).astype(np.float32)
name_index = {n:i for i,n in enumerate(names)}
assert baseline_vectors.shape==(82,768)

# Verify the actual loaded architecture. Sentence Transformers versions expose
# the CLS pooling setting in different forms. Never substitute a missing attribute
# with False: the official LaBSE checkpoint uses CLS pooling.
module_types = [type(model[i]).__name__ for i in range(len(model))]
expected_types = ["Transformer", "Pooling", "Dense", "Normalize"]
if module_types != expected_types:
    raise RuntimeError(f"Unexpected LaBSE modules: {module_types}")
pooling = model[1]
pooling_cfg = (pooling.get_config_dict()
               if callable(getattr(pooling, "get_config_dict", None)) else {})
runtime_mode = pooling_cfg.get("pooling_mode", getattr(pooling, "pooling_mode", None))
if runtime_mode is not None:
    if runtime_mode != "cls" and runtime_mode != ["cls"] and runtime_mode != ("cls",):
        raise RuntimeError(f"Expected CLS pooling, got {runtime_mode!r}")
elif "pooling_mode_cls_token" in pooling_cfg:
    if pooling_cfg["pooling_mode_cls_token"] is not True:
        raise RuntimeError(f"Expected CLS pooling: {pooling_cfg}")
elif getattr(pooling, "pooling_mode_cls_token", None) is False:
    raise RuntimeError("Loaded pooling is not CLS")
else:
    print("Pooling attribute unavailable: verifying numerical equivalence below.")
print("Verified modules:", module_types)
print("Runtime pooling configuration:", pooling_cfg or "not exposed")
for p in model[0].parameters():
    p.requires_grad_(False)
model[0].eval()
model[1].eval()
dense=model[2]
for p in dense.parameters():
    p.requires_grad_(True)

# Cache frozen BERT+Pooling output once. Training the actual LaBSE Dense
# projection now takes only seconds/minutes on CPU.
@torch.no_grad()
def get_pooled(texts):
    blocks=[]
    for start in range(0,len(texts),8):
        features = model.tokenize(texts[start:start+8])
        features = {k:v.to(model.device) if torch.is_tensor(v) else v
                    for k,v in features.items()}
        features = model[0](features)
        features = model[1](features)
        blocks.append(features["sentence_embedding"].detach().cpu())
    return torch.cat(blocks,dim=0)

pooled = get_pooled(names)
assert pooled.shape == (82,768)
with torch.no_grad():
    z = dense({"sentence_embedding":pooled.clone()})["sentence_embedding"]
    z = F.normalize(z,p=2,dim=1)
    discrepancy = float((z - torch.tensor(baseline_vectors)).abs().max())
print(f"Pretrained-output consistency (max absolute difference): {discrepancy:.6g}")
assert discrepancy < 0.002, "Unexpected encoder/pooling/projection mismatch. Stop before training."

# Weak labels: first two digits of SemelAnafSofi — economic sector, NOT occupation.
tidx={n:i for i,n in enumerate(train_names)}
train_positions=torch.tensor([name_index[n] for n in train_names])
training_features=pooled[train_positions]
train_base=F.normalize(torch.tensor(baseline_vectors)[train_positions],p=2,dim=1)
positive=[(tidx[a],tidx[b])
          for a,b in combinations(train_names,2)
          if stable_sector[a]==stable_sector[b]]
negative=[(tidx[a],tidx[b])
          for a,b in combinations(train_names,2)
          if stable_sector[a]!=stable_sector[b]]
assert positive and negative

# Mine high-similarity hard negatives strictly inside the train-only names.
ranked=sorted(negative,key=lambda p:float(train_base[p[0]] @ train_base[p[1]]),reverse=True)
selected_neg=ranked[:min(len(ranked),2*len(positive))]
remaining=ranked[len(selected_neg):]
if remaining:
    add=rng.choice(len(remaining),size=min(len(remaining),len(positive)),replace=False)
    selected_neg += [remaining[int(i)] for i in add]
pidx=torch.tensor(positive,dtype=torch.long)
nidx=torch.tensor(selected_neg,dtype=torch.long)
initial_state={k:v.detach().clone() for k,v in dense.state_dict().items()}
optimizer=torch.optim.AdamW(dense.parameters(), lr=8e-5,weight_decay=0.001)
loss_history=[]
for epoch in range(35):
    dense.train()
    optimizer.zero_grad(set_to_none=True)
    adapted=dense({"sentence_embedding":training_features.clone()})["sentence_embedding"]
    adapted=F.normalize(adapted,p=2,dim=1)
    sim=adapted @ adapted.T
    pos_loss=(1.0-sim[pidx[:,0],pidx[:,1]]).square().mean()
    neg_loss=F.relu(sim[nidx[:,0],nidx[:,1]]-0.35).square().mean()
    drift=(1-(adapted*train_base).sum(dim=1)).square().mean()
    loss=pos_loss+1.75*neg_loss+0.15*drift
    assert torch.isfinite(loss), "Non-finite training loss"
    loss.backward()
    torch.nn.utils.clip_grad_norm_(dense.parameters(),1.0)
    optimizer.step()
    loss_history.append([epoch+1,float(loss.detach()),float(pos_loss.detach()),float(neg_loss.detach())])

dense.eval()
model.eval()
adapted_vectors=model.encode(names,batch_size=8,device="cpu",
    normalize_embeddings=True,convert_to_numpy=True,
    show_progress_bar=False).astype(np.float32)
assert adapted_vectors.shape == baseline_vectors.shape and np.isfinite(adapted_vectors).all()
change=float(np.linalg.norm(adapted_vectors-baseline_vectors,axis=1).mean())
assert change>1e-7, "Fine-tuned weights did not change the outputs."
pd.DataFrame(loss_history,columns=["epoch","total_loss","positive_loss","negative_loss"]).to_csv(
    OUT/"training_curve.csv",index=False)
torch.save(dense.state_dict(),OUT/"labse_finetuned_dense_head.pt")
print(f"Fine-tuning completed on {len(positive)} positive and {len(selected_neg)} negative train-only pairs.")
print(f"Mean change in normalized embeddings: {change:.5f}. BERT backbone frozen; LaBSE Dense updated.")
print("Trained Dense weights:",OUT/"labse_finetuned_dense_head.pt")

## Held-out evaluation, complete-linkage clusters, and one-hot encoding

This section compares baseline and fine-tuned embeddings on employer names **not used in the fine-tuning supervision**. Positive relationships mean matching *economic-industry prefixes*, not identical companies or occupations. The pretrained and fine-tuned models use the same fixed **cosine > 0.50** decision rule.

For grouping, **complete linkage** avoids the transitivity problem of a loose similarity graph: every pair within any cluster must satisfy the threshold. This is stricter than merely connecting all edges above 0.50. Each survey record receives its employer cluster ID and a one-hot vector. A new name must exceed the threshold against **every member** of a cluster to join it, otherwise it is marked unknown. The original target `SemelMishlachSofi` is used only for descriptive occupation diversity, never for fine-tuning or cluster formation.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from itertools import combinations

# Held-out employers never appear in training pairs, so evaluation is name-disjoint.
validation_pairs=[(a,b,int(stable_sector[a]==stable_sector[b]))
                  for a,b in combinations(holdout,2)]
assert sum(y for _,_,y in validation_pairs)>0
threshold_grid = np.round(np.arange(0.30,0.901,0.05),2)

def pair_scores(emb):
    return np.asarray([float(emb[name_index[a]] @ emb[name_index[b]])
                       for a,b,_ in validation_pairs])

pre_scores=pair_scores(baseline_vectors)
post_scores=pair_scores(adapted_vectors)
truth=np.array([y for _,_,y in validation_pairs],dtype=np.int64)
records=[]
for stage, scores in [("Pretrained",pre_scores),("Dense fine-tuned",post_scores)]:
    for threshold in threshold_grid:
        predicted=(scores>threshold).astype(np.int64)
        tp=int(((truth==1)&(predicted==1)).sum())
        fp=int(((truth==0)&(predicted==1)).sum())
        fn=int(((truth==1)&(predicted==0)).sum())
        tn=int(((truth==0)&(predicted==0)).sum())
        precision=tp/(tp+fp) if tp+fp else float("nan")
        recall=tp/(tp+fn) if tp+fn else float("nan")
        f1=(2*precision*recall/(precision+recall) 
            if np.isfinite(precision) and np.isfinite(recall) and precision+recall else float("nan"))
        records.append(dict(stage=stage,threshold=float(threshold),TP=tp,FP=fp,FN=fn,TN=tn,
                            precision=precision,recall=recall,F1=f1))
metrics=pd.DataFrame(records)
metrics.to_csv(OUT/"heldout_sector_metrics.csv",index=False)
chosen=metrics[np.isclose(metrics.threshold,0.5)].copy()
print("Name-disjoint validation: similarity means same 2-digit economic INDUSTRY, NOT same occupation.")
display(chosen.round(4))
if np.isfinite(chosen.F1).all():
    gain = float(chosen.F1.iloc[1]-chosen.F1.iloc[0])
    print(f"Observed F1 change at >0.50: {gain:+.4f}. A negative value is deterioration.")
else:
    print("F1 undefined for at least one condition; no improvement claim.")
print(f"Validation: {len(holdout)} unseen employer strings, {len(validation_pairs)} independent name pairs.")
print("Sector labels may be noisy; this is an exploratory experiment on synthetic survey records.")

# Complete-linkage: avoid 'transitive-chain' merges that single-linkage could cause.
# For distance=1-cosine, complete-linkage guarantees within-cluster minimum cosine
# exceeds the chosen threshold (up to floating-point tolerance).
def cluster_at_half(emb):
    kwargs=dict(n_clusters=None,distance_threshold=0.5,linkage="complete")
    try:
        algorithm=AgglomerativeClustering(metric="cosine",**kwargs)
    except TypeError:
        algorithm=AgglomerativeClustering(affinity="cosine",**kwargs)
    labels=algorithm.fit_predict(emb)
    for cid in sorted(set(labels)):
        ids=np.flatnonzero(labels==cid)
        if len(ids)>1:
            cos=emb[ids]@emb[ids].T
            offdiag=cos[np.triu_indices(len(ids),1)]
            assert float(offdiag.min())>=0.49999, "Cluster violates pairwise >0.50 condition"
    return labels

pre_cluster=cluster_at_half(baseline_vectors)
post_cluster=cluster_at_half(adapted_vectors)
name_to_cluster=dict(zip(names,map(int,post_cluster)))
name_to_before=dict(zip(names,map(int,pre_cluster)))
survey["cluster_pre"]=survey.ShemAvoda.map(name_to_before)
survey["cluster_post"]=survey.ShemAvoda.map(name_to_cluster)

# One-Hot features per record. Unknown categories during transform do not crash.
try:
    encoder=OneHotEncoder(handle_unknown="ignore",sparse_output=True,dtype=np.float32)
except TypeError:  # Fallback for older but still capable scikit-learn versions
    encoder=OneHotEncoder(handle_unknown="ignore",sparse=True,dtype=np.float32)
encoded=encoder.fit_transform(survey[["cluster_post"]])
feat_names=[f"ShemAvoda_cluster_{int(i)}" for i in encoder.categories_[0]]
feature_table=pd.DataFrame(encoded.toarray(),columns=feat_names)
features=pd.concat([survey[["ID","ShemAvoda","cluster_pre","cluster_post"]].reset_index(drop=True),
                    feature_table],axis=1)
assert len(features)==200 and encoded.shape==(200,len(encoder.categories_[0]))
assert np.all(encoded.toarray().sum(axis=1)==1)
features.to_csv(OUT/"survey_employer_cluster_onehot.csv",index=False,encoding="utf-8-sig")
report=pd.DataFrame({
    "ShemAvoda":names,"industry_proxy":[stable_sector.get(n,"MIXED/UNKNOWN") for n in names],
    "cluster_before":pre_cluster.astype(int),"cluster_after":post_cluster.astype(int),
    "survey_records":[int((survey.ShemAvoda==n).sum()) for n in names]
})
report.to_csv(OUT/"employer_cluster_membership.csv",index=False,encoding="utf-8-sig")
print(f"Complete-linkage clusters @ >0.50: before={len(set(pre_cluster))}, after={len(set(post_cluster))}.")
print(f"One-hot shape: {encoded.shape} | all 200 survey records represented.")
display(report.sort_values(["cluster_after","ShemAvoda"]).head(24))

# A new employer receives an existing cluster only if its similarity to EVERY
# member of that cluster is strictly above 0.50; otherwise unknown=-1.
def transform_new_employers(new_names):
    z = model.encode(new_names,normalize_embeddings=True,
                     convert_to_numpy=True,show_progress_bar=False)
    assigned=[]
    for v in z:
        possible=[]
        for cid in sorted(set(post_cluster)):
            members=adapted_vectors[np.asarray(post_cluster)==cid]
            scores=members @ v
            if np.all(scores>0.50):
                possible.append((float(scores.mean()),int(cid)))
        assigned.append(max(possible)[1] if possible else -1)
    onehot=encoder.transform(np.asarray(assigned).reshape(-1,1))
    return assigned,onehot

# Descriptive occupation diversity: never used to train the LaBSE projection.
diversity=(survey.groupby("cluster_post").SemelMishlachSofi.nunique()
           .rename("distinct_occupation_codes").reset_index())
print("Occupation-code diversity within employer clusters (descriptive only):")
display(diversity.sort_values("distinct_occupation_codes",ascending=False).head(10))
print("CLUSTERS_AND_ONEHOT_SUCCESS")

## Visual evidence studio — interactive before/after, semantic flow, threshold curves

All figures below are generated from **actual measured** baseline and fine-tuned vectors during **Run all**. The network uses **NetworkX's existing `spring_layout(dim=3)` force-directed layout**, calculated once on the union of before/after cosine connections and reused for both frames. Node positions describe graph topology, **not 768-dimensional cosine distance**. Lines represent above-0.50 cosine connections and are limited to the strongest connections for legibility; complete-linkage clustering uses the **entire** matrix. Node color indicates a weak two-digit economic-sector code, not a validated occupation or a verified employer identity.

The animated semantic constellation has **Before/After controls**, free 3D rotation and hover, and a **light/dark theme toggle** in the portfolio palette. Other interactive charts show held-out threshold performance, how similarities change, and the flow of names between clustering results. There are no decorative or invented data values.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter,defaultdict
import networkx as nx
from itertools import combinations

base=THEME
colors=THEME["series"]
freq=Counter(survey.ShemAvoda)
industry_for=[stable_sector.get(n, "MIXED") for n in names]
distinct_industry=sorted(set(industry_for))
groupcolor={sec:colors[i%len(colors)] for i,sec in enumerate(distinct_industry)}
node_colors=[groupcolor[sec] if sec!="MIXED" else "#9A9C98" for sec in industry_for]
node_size=[max(5,min(17,5+2.5*np.sqrt(freq[n]))) for n in names]

# Native NetworkX spring_layout(dim=3), Fruchterman-Reingold automatic positioning.
# Build one UNION graph from the actual before/after >0.50 relations so
# the layout is identical in both frames. Only the measured edges change.
cosine_before = baseline_vectors @ baseline_vectors.T
cosine_after = adapted_vectors @ adapted_vectors.T
n_nodes = len(names)
assert cosine_before.shape == cosine_after.shape == (n_nodes, n_nodes)
union_graph = nx.Graph()
union_graph.add_nodes_from(range(n_nodes))
for i in range(n_nodes):
    for j in range(i + 1, n_nodes):
        strength = max(float(cosine_before[i, j]), float(cosine_after[i, j]))
        if strength > 0.50:
            union_graph.add_edge(i, j, weight=strength - 0.48)
layout_3d = nx.spring_layout(union_graph, dim=3, weight="weight",
                             iterations=180, seed=2026)
shared_positions_3d = np.vstack([layout_3d[i] for i in range(n_nodes)])
assert shared_positions_3d.shape == (82, 3) and np.isfinite(shared_positions_3d).all()
pos_before = shared_positions_3d
pos_after = shared_positions_3d
print(f"Automatic 3D graph layout: {n_nodes} nodes, {union_graph.number_of_edges()} unique edges.")
print("Same positions before/after; original 768D cosine determines all relations.")

def strongest_edges(mat, limit=120):
    scores=mat@mat.T
    items=[]
    used=set()
    for i in range(len(names)):
        inds=np.argsort(scores[i])[::-1]
        count=0
        for j in inds:
            if i==j or scores[i,j]<=0.50: continue
            edge=tuple(sorted((i,int(j))))
            if edge not in used:
                items.append((float(scores[i,j]),*edge))
                used.add(edge)
            count+=1
            if count>=3: break
    return sorted(items,reverse=True)[:limit]

def constellation(points,vecs,clusters):
    linesx,linesy,linesz=[],[],[]
    for weight,i,j in strongest_edges(vecs):
        linesx.extend([points[i,0],points[j,0],None])
        linesy.extend([points[i,1],points[j,1],None])
        linesz.extend([points[i,2],points[j,2],None])
    edges=go.Scatter3d(x=linesx,y=linesy,z=linesz,mode="lines",showlegend=False,
        line=dict(color=THEME["muted"],width=2),opacity=0.23,hoverinfo="skip")
    info=[f"{name}<br>Records: {freq[name]}<br>Industry prefix: {industry_for[k]}"
          f"<br>Cluster: {int(clusters[k])}" for k,name in enumerate(names)]
    nodes=go.Scatter3d(x=points[:,0],y=points[:,1],z=points[:,2],
        mode="markers",showlegend=False,text=info,hovertemplate="%{text}<extra></extra>",
        marker=dict(color=node_colors,size=node_size,opacity=0.96,
                    line=dict(color=THEME["paper"],width=0.6)))
    return [edges,nodes]

frame_pre=constellation(pos_before,baseline_vectors,pre_cluster)
frame_post=constellation(pos_after,adapted_vectors,post_cluster)
fig3=go.Figure(data=frame_pre,frames=[
    go.Frame(data=frame_pre,name="PRETRAINED"),
    go.Frame(data=frame_post,name="FINE-TUNED")])
fig3.update_layout(
    title=dict(text="<b>SEMANTIC CONSTELLATIONS</b><br><sup>Native NetworkX auto layout · edges scored in original 768D · hover to explore</sup>",
               x=0.045,font=dict(size=24)),
    height=760,paper_bgcolor=THEME["paper"],font=dict(color=THEME["ink"],size=13),
    margin=dict(l=10,r=10,t=104,b=28),
    scene=dict(bgcolor=THEME["paper"],
       xaxis=dict(visible=False),yaxis=dict(visible=False),zaxis=dict(visible=False),
       aspectmode="data",camera=dict(eye=dict(x=1.7,y=1.55,z=1.2))),
    updatemenus=[
      dict(type="buttons",direction="right",showactive=True,x=0.02,y=1.055,
        buttons=[
         dict(label="BEFORE",method="animate",args=[["PRETRAINED"],
              dict(mode="immediate",frame=dict(duration=750,redraw=True),transition=dict(duration=700))]),
         dict(label="AFTER",method="animate",args=[["FINE-TUNED"],
              dict(mode="immediate",frame=dict(duration=750,redraw=True),transition=dict(duration=700))]),
         dict(label="PLAY",method="animate",args=[None,
              dict(mode="immediate",frame=dict(duration=950,redraw=True),transition=dict(duration=650))])
        ]),
      dict(type="buttons",direction="right",showactive=True,x=0.74,y=1.055,
        buttons=[
          dict(label="LIGHT",method="relayout",args=[{
              "paper_bgcolor":THEME["paper"],"font.color":THEME["ink"],
              "scene.bgcolor":THEME["paper"]}]),
          dict(label="DARK",method="relayout",args=[{
              "paper_bgcolor":DARK["paper"],"font.color":DARK["ink"],
              "scene.bgcolor":DARK["paper"]}])
        ])
    ],
    sliders=[dict(active=0,x=0.18,len=0.68,y=0.015,pad=dict(t=30),
       currentvalue=dict(prefix="Representation: ",font=dict(size=14)),
       steps=[dict(label=lab,method="animate",args=[[key],dict(
           mode="immediate",frame=dict(duration=750,redraw=True),
           transition=dict(duration=700))])
           for lab,key in [("Pretrained","PRETRAINED"),("Fine-tuned","FINE-TUNED")]])]
)
fig3.show()
fig3.write_html(OUT/"01_semantic_constellations_interactive.html",include_plotlyjs=True)

# Test-set performance. These scores are on weak ECONOMIC-SECTOR labels,
# not the occupational target or identity of the company.
metric_chart=make_subplots(rows=1,cols=3,subplot_titles=["Precision","Recall","F1"])
for col,measure in enumerate(("precision","recall","F1"),1):
    for stage,c,style in [("Pretrained",THEME["green"],"solid"),
                          ("Dense fine-tuned",THEME["chart_series"] if "chart_series" in THEME else colors[1],"dash")]:
        part=metrics[metrics.stage==stage]
        metric_chart.add_trace(go.Scatter(
            x=part.threshold,y=part[measure],name=stage,legendgroup=stage,
            showlegend=(col==1),mode="lines+markers",
            line=dict(color=c,width=3,dash=style),marker=dict(size=8),
            hovertemplate=f"{measure}=%{{y:.2%}}<br>Threshold=%{{x:.2f}}<extra>{stage}</extra>"),
            row=1,col=col)
    metric_chart.add_vline(x=0.50,line_dash="dot",line_color=colors[3],row=1,col=col)
    metric_chart.update_yaxes(range=[0,1.02],tickformat=".0%",row=1,col=col)
metric_chart.update_layout(title="<b>THRESHOLD OBSERVATORY</b><br><sup>Held-out employer names · economic-sector proxy · 0.50 pre-specified</sup>",
    font=dict(color=THEME["ink"]),paper_bgcolor=THEME["paper"],plot_bgcolor=THEME["paper"],
    height=460,margin=dict(l=50,r=50,t=100,b=60),legend=dict(orientation="h",y=-0.17))
metric_chart.show()
metric_chart.write_html(OUT/"02_heldout_threshold_observatory.html",include_plotlyjs=True)

# Matrix of original cosine, adapted cosine, and their difference on common labels.
focus=sorted(names,key=lambda n:(-freq[n],n))[:18]
for agr in ("גידול תפוזים ","גידול תפוחים "):
    if agr in names and agr not in focus: focus.append(agr)
ii=[name_index[n] for n in focus]
cos0=baseline_vectors[ii]@baseline_vectors[ii].T
cos1=adapted_vectors[ii]@adapted_vectors[ii].T
delta=cos1-cos0
figheat=go.Figure(data=[
  go.Heatmap(x=focus,y=focus,z=cos0,zmin=-1,zmax=1,colorscale="Greens",visible=True,
      colorbar=dict(title="cosine"),hovertemplate="%{y}<br>%{x}<br>cos=%{z:.3f}<extra></extra>"),
  go.Heatmap(x=focus,y=focus,z=cos1,zmin=-1,zmax=1,colorscale="Greens",visible=False,
      colorbar=dict(title="cosine"),hovertemplate="%{y}<br>%{x}<br>cos=%{z:.3f}<extra></extra>"),
  go.Heatmap(x=focus,y=focus,z=delta,zmid=0,colorscale=[
      [0,colors[3]],[0.5,THEME["paper"]],[1,THEME["green"]]],visible=False,
      colorbar=dict(title="Δ cosine"),hovertemplate="%{y}<br>%{x}<br>Δ=%{z:+.3f}<extra></extra>")
])
figheat.update_layout(
 title="<b>SIMILARITY LABORATORY</b><br><sup>Actual 768D cosine values · toggle before, after, change</sup>",
 height=800,paper_bgcolor=THEME["paper"],plot_bgcolor=THEME["paper"],font=dict(color=THEME["ink"],size=12),
 margin=dict(l=140,r=90,t=110,b=155),
 xaxis=dict(tickangle=45,automargin=True),yaxis=dict(autorange="reversed",automargin=True),
 updatemenus=[dict(type="buttons",direction="right",x=0.05,y=1.06,buttons=[
    dict(label=lab,method="update",args=[{"visible":[i==j for i in range(3)]},
                                          {"title.text":"<b>SIMILARITY LABORATORY — "+lab+"</b>"}])
    for j,lab in enumerate(["BEFORE","AFTER","CHANGE"])])])
figheat.show()
figheat.write_html(OUT/"03_similarity_laboratory.html",include_plotlyjs=True)

# Sector summaries: measured from actual source, not assumed category names.
def cluster_label(groups,cid,prefix):
    members=[names[i] for i,c in enumerate(groups) if int(c)==int(cid)]
    dominant=sorted(members,key=lambda n:(-freq[n],n))[0]
    return f"{prefix}{cid}: {dominant} ({len(members)})"

counts0=Counter(map(int,pre_cluster))
counts1=Counter(map(int,post_cluster))
keep0=set(c for c,n in counts0.most_common(10))
keep1=set(c for c,n in counts1.most_common(10))
def gname(c,keep):
    return str(c) if c in keep else "OTHER"
flows=Counter((gname(int(a),keep0),gname(int(b),keep1)) for a,b in zip(pre_cluster,post_cluster))
left=sorted(set(a for a,b in flows))
right=sorted(set(b for a,b in flows))
keys=[("PRE",i) for i in left]+[("POST",i) for i in right]
loc={key:i for i,key in enumerate(keys)}
labels=[("Before other clusters" if i=="OTHER" else cluster_label(pre_cluster,int(i),"Before C"))
        for i in left]+[
        ("After other clusters" if i=="OTHER" else cluster_label(post_cluster,int(i),"After C"))
        for i in right]
figflow=go.Figure(data=[go.Sankey(
 arrangement="snap",node=dict(label=labels,pad=14,thickness=13,color=THEME["green"]),
 link=dict(source=[loc[("PRE",a)] for a,b in flows],
           target=[loc[("POST",b)] for a,b in flows],
           value=[v for v in flows.values()],
           color="rgba(80,117,137,0.27)")
)])
figflow.update_layout(title="<b>CLUSTER MIGRATION</b><br><sup>How the 82 original employer strings regroup after supervised Dense tuning</sup>",
    height=700,paper_bgcolor=THEME["paper"],font=dict(color=THEME["ink"],size=11),
    margin=dict(l=12,r=12,t=100,b=24))
figflow.show()
figflow.write_html(OUT/"04_cluster_migration.html",include_plotlyjs=True)
print("INTERACTIVE_VISUALIZATIONS_SUCCESS — 4 self-contained HTML figures generated.")


## Native 8K exports — both PORTFOLIO LIGHT and PORTFOLIO DARK

The final cell creates two **7680 × 4320 pixel** static 8K portfolio posters from the **same measured** baseline and fine-tuned cosine graphs using a single NetworkX automatic force-directed layout, plus self-contained Plotly HTML files, the fine-tuned Dense checkpoint, cluster assignments, a CSV of **200 survey records with one-hot features**, training curves, held-out threshold metrics, and a manifest explaining the experiment's scope. Results are packaged into one ZIP in the free Colab runtime. No browser, screenshot service, paid rendering engine, prompt, or token is required. Static 8K plots are accompanied by the interactive 3D graphics above, which can be freely rotated and inspected.

**Scientific distinction:** The fine-tuning optimizes resemblance according to a **weak economic-sector proxy**, not identical employers or occupation labels. The 0.50 threshold is fixed by the research question, not established as universal. The test does not assert an improvement unless held-out metrics actually show one. The resulting one-hot clusters are **experimental features**, not validated occupation classifications.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from PIL import Image as PILImage
from IPython.display import Image as NotebookImage, display, FileLink
import json, zipfile, hashlib

# NetworkX + Matplotlib native graph drawing, no custom coordinate placement.
# This cell independently computes its graph positions; it does NOT rely on
# variables created by the preceding interactive visualization cell.
assert len(names)==len(baseline_vectors)==len(adapted_vectors)==82
pre_cosine = baseline_vectors @ baseline_vectors.T
post_cosine = adapted_vectors @ adapted_vectors.T
g_auto = nx.Graph()
g_auto.add_nodes_from(range(len(names)))
for i in range(len(names)):
    for j in range(i+1,len(names)):
        strength=max(float(pre_cosine[i,j]),float(post_cosine[i,j]))
        if strength>0.50:
            g_auto.add_edge(i,j,weight=strength-0.48)
# Native Fruchterman-Reingold solver: same deterministic positions in both views.
positions = nx.spring_layout(g_auto, dim=2, weight="weight",
                             iterations=180, seed=2026)
assert len(positions)==len(names)
freq_local=Counter(survey.ShemAvoda)
sectors_local=[stable_sector.get(n,"MIXED/UNKNOWN") for n in names]
unique_sectors_local=sorted(set(sectors_local))
sector_palette_local={s:THEME["series"][i%len(THEME["series"])]
                      for i,s in enumerate(unique_sectors_local)}
colors_local=[sector_palette_local[s] if s!="MIXED/UNKNOWN" else "#999A95"
              for s in sectors_local]
sizes_local=[85+60*np.sqrt(int(freq_local[n])) for n in names]

def select_visible_edges(score_matrix, maximum=170):
    # All edges are considered by the native layout, while the poster only
    # draws the strongest per-node connections to remain legible at 8K.
    ranked=sorted(
        [(float(score_matrix[i,j]),i,j)
         for i,j in g_auto.edges() if score_matrix[i,j] > 0.50],
        reverse=True)
    selected=[]; degree=Counter()
    for sim,i,j in ranked:
        if len(selected)>=maximum: break
        if degree[i]>=5 or degree[j]>=5: continue
        selected.append((i,j))
        degree[i]+=1; degree[j]+=1
    return selected

# Native, measured 8K PNG = 32 in × 18 in × 240 DPI (7680 × 4320).
def poster(mode):
    dark=(mode=="dark")
    paper=DARK["paper"] if dark else THEME["paper"]
    ink=DARK["ink"] if dark else THEME["ink"]
    edge_color="#91B6A4" if dark else THEME["muted"]
    fig,axes=plt.subplots(1,2,figsize=(32,18),dpi=240,facecolor=paper)
    for ax,score_matrix,clusters,title in [
        (axes[0],pre_cosine,pre_cluster,"01  PRETRAINED LaBSE"),
        (axes[1],post_cosine,post_cluster,"02  DENSE FINE-TUNED")
    ]:
        ax.set_facecolor(paper)
        nx.draw_networkx_edges(
            g_auto,pos=positions,edgelist=select_visible_edges(score_matrix),
            ax=ax,edge_color=edge_color,alpha=0.20,width=0.9)
        nx.draw_networkx_nodes(
            g_auto,pos=positions,ax=ax,nodelist=list(g_auto.nodes),
            node_color=colors_local,node_size=sizes_local,
            edgecolors=paper,linewidths=0.85,alpha=0.95)
        ax.set_axis_off()
        ax.set_title(title,fontweight="bold",fontsize=30,color=ink,pad=30,loc="left")
        ax.text(0.02,0.025,
            f"{len(set(clusters))} complete-linkage clusters  |  {len(names)} employer labels",
            color=ink,fontsize=19,transform=ax.transAxes)
    fig.suptitle("THE SEMANTIC WORKPLACE ATLAS",x=0.06,y=0.954,
                 ha="left",fontsize=45,fontweight="bold",color=ink)
    fig.text(0.06,0.914,
        "BEFORE / AFTER     ·     One NetworkX auto layout, two measured similarity graphs"
        "     ·     LaBSE economic-sector proxy",
        fontsize=19,color=ink)
    fig.text(0.06,0.063,
        "Source: 200 simulated survey records, 82 employer labels  |  "
        "connections: 768D cosine > 0.50  |  palette: original portfolio",
        fontsize=17,color=ink)
    fig.text(0.06,0.041,
        "Node proximity reflects automatic force layout, NOT exact cosine distance. "
        "Only strong edges are drawn for clarity; all pairs form the clusters.",
        fontsize=15,color=ink)
    fig.subplots_adjust(left=.04,right=.96,top=.855,bottom=.115,wspace=.12)
    file=OUT/f"05_semantic_workplace_atlas_8k_{mode}.png"
    fig.savefig(file,dpi=240,facecolor=paper,format="png")
    plt.close(fig)
    with PILImage.open(file) as im:
        assert im.size==(7680,4320),f"Unexpected poster dimensions: {im.size}"
        thumb=im.copy()
        thumb.thumbnail((1280,720))
        thumb.save(OUT/f"05_semantic_workplace_atlas_preview_{mode}.png")
    return file

poster_light=poster("light")
poster_dark=poster("dark")
display(HTML("<h3>True 8K exports · light and dark</h3>"))
display(NotebookImage(filename=str(OUT/"05_semantic_workplace_atlas_preview_light.png")))
display(NotebookImage(filename=str(OUT/"05_semantic_workplace_atlas_preview_dark.png")))

report_summary={
 "source":"Original simulated CBS survey workbook, 200 ID-linked records",
 "source_columns":["ID","ShemAvoda","SemelAnafSofi","SemelMishlachSofi"],
 "snapshot_sha256": DATA_SHA256,
 "model":"sentence-transformers/LaBSE",
 "architecture":"BERT encoder frozen; existing LaBSE Dense module fine-tuned (PyTorch)",
 "objective":"Weak economic-sector similarity from two-digit SemelAnafSofi prefix, NOT occupation or employer identity",
 "target_policy":"SemelMishlachSofi is NEVER used during projection fine-tuning or clustering",
 "group_split":"Unique employer strings held out by name before any supervised pair training",
 "threshold":"strict cosine similarity > 0.50 (predefined, not optimized on held-out data)",
 "clustering":"AgglomerativeClustering metric=cosine linkage=complete; pairwise cluster-member threshold checked",
 "unknown_label_policy":"A new name joins a cluster only if all member cosine scores >0.50; else -1",
 "validation_positive_pairs":int(truth.sum()),
 "validation_negative_pairs":int((1-truth).sum()),
 "training_names":len(train_names),
 "heldout_names":len(holdout),
 "excluded_supervision_names":len(uncertain_names),
 "clusters_before":len(set(pre_cluster)),
 "clusters_after":len(set(post_cluster)),
 "onehot_shape":list(encoded.shape),
 "results_at_0_50":chosen.to_dict(orient="records"),
 "plot_style_source":"Advance-Data-Science/theme/portfolio/palette.json",
 "limitations":[
    "Only 200 simulated records and potentially noisy manually generated industry codes.",
    "Industry sector similarity is not proof of same employer nor same occupation.",
    "No demonstrated increase in occupation classification accuracy.",
    "Validation metrics are based on related pairs from a small held-out-name set.",
    "Zero false positives on this sample would not prove a universally safe 0.50 threshold.",
    "3D/2D PCA reduces 768 dimensions and cannot preserve every original cosine relation.",
    "A trained Dense checkpoint must be loaded into the matching base LaBSE model for reuse."
 ]
}
(OUT/"run_manifest.json").write_text(
    json.dumps(report_summary,ensure_ascii=False,indent=2,default=float),encoding="utf-8")

bundle=Path("/content/labse_occupation_finetuning_results.zip")
with zipfile.ZipFile(bundle,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for file in sorted(OUT.iterdir()):
        if file.is_file(): z.write(file,arcname=file.name)
print(f"8K exports: {poster_light.name}, {poster_dark.name}")
print(f"Evidence files: {len(list(OUT.iterdir()))}; ZIP size: {bundle.stat().st_size/1e6:.1f} MB")
print("All generated results were computed within this Colab execution.")
display(FileLink(str(bundle)))
print("ONE_CLICK_COLAB_COMPLETE")

**Interpretation rule:** A successful run proves that the pipeline produces real embeddings and similarity scores. It does **not** establish the precision of multilingual company-name matching. Decide whether to use this representation only after evaluating positive and negative pairs from the actual `ShemAvoda` distribution, including ambiguous names and generic employer descriptions. The notebook intentionally performs no automatic name replacement.

### Research and implementation references

- [Original LaBSE weights and modules](https://huggingface.co/sentence-transformers/LaBSE/tree/main) — 12-layer BERT, CLS pooling, trained Dense projection, normalization.
- [LaBSE official pooling settings](https://huggingface.co/sentence-transformers/LaBSE/blob/main/1_Pooling/config.json) — CLS, not mean pooling.
- [Sentence Transformers training losses](https://www.sbert.net/docs/package_reference/sentence_transformer/losses.html) — contrastive / supervised metric learning.
- [scikit-learn complete-link clustering](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html) — max pairwise distance prevents chaining.
- [scikit-learn one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) — categorical feature mapping with unknown handling.
- [Plotly 3D scientific visualization](https://plotly.com/python/3d-charts/), [animations](https://plotly.com/python/animations/), and [templates](https://plotly.com/python/templates/) — reproducible interactive charts.
- [Source design system](https://github.com/shulruferor-sudo/Advance-Data-Science/tree/main/theme/portfolio) — portfolio palette and typography.

All diagrams are computed from observed embeddings. The two 8K PNG images are generated only **after** the Colab cells execute successfully; no results are embedded or implied by the mere presence of code.

**Methodological status:** The experiment runs on all 200 simulated records and exports cluster one-hot columns. This export is an exploratory **feature-engineering artifact**, **not** a leakage-safe estimate of occupation prediction. If these features are incorporated into the actual `SemelMishlachSofi` classifier, **name splitting, weak-label fine-tuning, clustering, and one-hot fitting must all occur inside each training fold**, with held-out records transformed by those fitted operations only. The industry code is available in the supplied training data but may not be available at production prediction time. The experiment does not establish that the extra features improve occupation prediction.

**Outputs:** Four interactive HTML graphics, two 8K posters (portfolio light/dark), trained LaBSE Dense weights, 200-row one-hot feature table, cluster memberships, training/evaluation CSVs and an audit manifest. All output files are created by Colab **Run all**, not precomputed in this GitHub notebook.